## C1 - Autonomous Behavior: Stopping
Author: George Gorospe, george.gorospe@nmaia.net\
Last Update: July 22, 2026

### About: In C0 you watched your Free/Blocked model make predictions while you drove. Today your robot acts on those predictions itself, for the first time -- no gamepad, no hands on the controller. This is your first autonomous behavior.

### Reading a Prediction in Code
### Back in C0, every time the camera captured a new frame, your model produced a prediction -- a class name (`'free'` or `'blocked'`) and a confidence score. In code, that's just a string and a number, like any other variable:
```python
label, confidence = predict_image(model, frame, class_names, device)
# label is 'free' or 'blocked'
```
### Today, instead of just displaying that label, your robot is going to *act* on it.

### Turning a Prediction Into a Decision
### The pattern is one you already know: an `if` / `else` statement.
```python
if label == 'blocked':
    # do something
else:
    # do something else
```
### The only new part is what goes inside each branch -- and that's what you'll write today.

### Why We Draw a Flowchart First
### Before writing the `if`/`else` above, it helps to draw out the decision as a simple flowchart: what does the robot sense, what question does it ask, and what does it do for each possible answer? A flowchart forces you to define every branch *before* you're debugging code -- much easier to fix a flowchart than to fix code that's already running on a moving robot.

### 🤖 Real Robotics Engineering: Flowcharts & State Machines
### This isn't just a classroom habit -- professional robotics and safety-critical systems are designed as flowcharts and **state machines** before a single line of code is written. You're doing what actual robotics teams do first.

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY C1.1: Stop When Blocked</span>

### Behavior Spec: Stop When Blocked
### The simplest possible autonomous behavior:
### - If the path is **free**, keep driving forward.
### - If the path is **blocked**, stop.

### That's the whole flowchart: one question, two branches.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP 1. Import required libraries

from sphero_sdk import RawMotorModesEnum
import ipywidgets as widgets
from IPython.display import display
from ipyfilechooser import FileChooser

from jetcam_lite import TraitletCamera, bgr8_to_jpeg

from robot_utils import get_rvr, close_if_exists
from jupyter_utils import register_dlink
from inference_utils import load_model_and_metadata
from behavior_utils import create_start_stop_buttons

### STEP 2. Choose and load your Free/Blocked model -- same as C0. Use the file browser to navigate to your `Models` folder and select a `.pth` file.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

model_chooser = FileChooser('/home/explorer/Models/')
model_chooser.filter_pattern = '*.pth'
display(model_chooser)

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP 3. Load the model you selected above.

model, device, class_names, training_record = load_model_and_metadata(model_chooser.selected)

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP 4. Connect to the robot and start the camera.

rvr = get_rvr()

camera = TraitletCamera()
camera.start()

image_widget = widgets.Image(format='jpeg', width=camera.width, height=camera.height)
register_dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)
display(image_widget)

### STEP 5. Now write the decision logic -- the actual behavior. Fill in both branches below: what should the robot do when the path is `'blocked'`, and what should it do when the path is `'free'`?
### Reminder: the RVR stops itself if it doesn't get a new drive command within about 2 seconds. You don't need to worry about that here -- this function gets called repeatedly (about twice a second) for as long as the behavior is running, so a `'free'` decision that drives forward will keep the robot moving on its own.

In [ ]:
#### ------> ACTIVITY C1.1: decide_action function <-----#####
# About: this function is called repeatedly while the behavior is running.
# Each time, it receives the model's current prediction as `label` and
# should command the robot accordingly.

##### INSTRUCTIONS: #####
# 1. In the 'blocked' branch, command the robot to stop (duty cycle 0).
# 2. In the 'free' branch, command the robot to drive forward.
# 3. Use rvr.raw_motors(), just like you did in A1.

def decide_action(rvr, label):
    if label == 'blocked':
        #<<<<<< replace this with your code >>>>>>
        pass

    else:  # label == 'free'
        #<<<<<< replace this with your code >>>>>>
        pass

### STEP 6. Build the Start/Stop buttons. Notice: running this cell does **not** move the robot -- nothing happens until you press Start.

######## WILL CAUSE ROBOT MOTION ONCE STARTED: ENSURE ROBOT IS ON THE GROUND, WITH A CLEAR PATH IN FRONT OF IT #########

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

buttons = create_start_stop_buttons(rvr, camera, model, class_names, device, decide_action)
display(buttons)

### Press **Start**, then hold an obstacle in front of your robot (or drive it toward a wall by hand) and watch what happens. Press **Stop** any time to immediately halt the behavior.

<span style="color: green; font-size: 55px; font-style: italic;">Student Discussion Time</span>

### Talk through these questions with your team:
### - Did it work as expected? Any false stops (stopped when the path was actually clear) or false continues (kept going when it should have stopped)?
### - Was there a delay between an obstacle appearing and the robot stopping? Where do you think that delay comes from?
### - Right now, the robot acts on every prediction equally, even ones the model wasn't very confident about. What do you think would happen if you only acted on predictions above, say, 90% confidence? Would that make the robot safer, or just slower to react?
### - Stopping is safe -- but is it useful on its own? What's missing?

## Nice work -- your robot just made its first autonomous decision, no hands on the controller.

## **NEXT**: In C2, instead of just stopping, your robot will turn and keep going.

# SOLUTIONS:

### ------> SOLUTION - Activity C1.1: decide_action function <-----

In [ ]:
#### ------> SOLUTION - Activity C1.1: decide_action function <-----#####

def decide_action(rvr, label):
    if label == 'blocked':
        # GEORGE's Solution:
        rvr.raw_motors(
            left_mode=RawMotorModesEnum.forward.value,
            left_duty_cycle=0,
            right_mode=RawMotorModesEnum.forward.value,
            right_duty_cycle=0
        )

    else:  # label == 'free'
        # GEORGE's Solution:
        rvr.raw_motors(
            left_mode=RawMotorModesEnum.forward.value,
            left_duty_cycle=100,
            right_mode=RawMotorModesEnum.forward.value,
            right_duty_cycle=100
        )

### Wrapping Up
Before you move on to the next notebook, run the cell below to release the robot's connection. This keeps things clean for whichever notebook you open next.

In [ ]:
#### ------> RUN THIS CELL WHEN YOU'RE DONE WITH THIS NOTEBOOK <-----#####
close_if_exists()
print("Robot connection closed. Safe to move on to the next notebook!")